In [2]:
!pip install -q -U pypdf langchain langchain-community langchain-core langchain-huggingface langchain-google-genai langchain-text-splitters faiss-cpu sentence-transformers

In [3]:
import os
from getpass import getpass
from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API Key: ")

print("Libraries imported and API key configured!")

/tmp/ipykernel_1062/1514298561.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Enter Gemini API Key: ··········
Libraries imported and API key configured!


In [4]:
# Upload PDF files
uploaded = files.upload()

if not uploaded:
    raise ValueError("Please upload at least one PDF file.")

# Load PDF documents
pages = []

for filename in uploaded.keys():
    if filename.lower().endswith(".pdf"):
        loader = PyPDFLoader(filename)
        pages.extend(loader.load())

if not pages:
    raise ValueError("No valid PDF files found.")

print("Total Pages Loaded:", len(pages))


# Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

chunks = splitter.split_documents(pages)

print("Total Chunks Created:", len(chunks))


# Create Hugging Face embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# Create FAISS vector database
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS Vector Database Created Successfully!")

Saving python_programming_rag_sample.pdf to python_programming_rag_sample.pdf
Total Pages Loaded: 4
Total Chunks Created: 10


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS Vector Database Created Successfully!


In [5]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0
)

print("Gemini Model Connected Successfully!")

Gemini Model Connected Successfully!


In [6]:

# Helper function to extract text from Gemini response

def extract_text(response):
    content = response.content

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        result = []

        for block in content:
            if isinstance(block, dict):
                result.append(block.get("text", ""))
            elif isinstance(block, str):
                result.append(block)

        return "\n".join(result)

    return str(content)


# AGENT 1: QUERY ANALYZER

def query_analyzer(question):

    prompt = f"""
    You are a query analysis agent.

    Understand the user's question.
    Generate 3 different short search queries
    that can help retrieve relevant PDF information.

    Return only 3 queries, one per line.

    User Question: {question}
    """

    response = llm.invoke(prompt)

    queries = [
        line.strip("- *0123456789. ")
        for line in extract_text(response).splitlines()
        if line.strip()
    ]

    queries = [q for q in queries if q]

    return (queries[:3] or [question])


# AGENT 2: DOCUMENT RETRIEVER

def document_retriever(queries):

    retrieved_docs = []
    seen = set()

    for query in queries:

        results = vectorstore.similarity_search(
            query,
            k=3
        )

        for doc in results:

            text = doc.page_content

            if text not in seen:
                seen.add(text)
                retrieved_docs.append(doc)

    return retrieved_docs


# Convert retrieved documents into context

def prepare_context(documents):

    context_parts = []

    for doc in documents:

        filename = os.path.basename(
            doc.metadata.get("source", "PDF")
        )

        page = doc.metadata.get("page", 0) + 1

        context_parts.append(
            f"[{filename} p.{page}]\n{doc.page_content}"
        )

    return "\n\n".join(context_parts)


# AGENT 3: ANSWER GENERATOR

def answer_generator(question, context):

    prompt = f"""
    You are an intelligent PDF question-answering agent.

    Answer the user's question using ONLY the
    retrieved PDF context.

    Instructions:
    1. Give a clear and simple answer.
    2. Do not invent information.
    3. Include source citations like [filename p.2].
    4. If the answer is unavailable, say:
       No relevant answer found in the PDF.

    PDF Context:
    {context}

    User Question:
    {question}

    Answer:
    """

    response = llm.invoke(prompt)

    return extract_text(response)


# AGENT 4: ANSWER REVIEWER

def answer_reviewer(question, context, draft):

    prompt = f"""
    You are an answer verification agent.

    Check whether the generated answer is supported
    by the retrieved PDF context.

    Instructions:
    1. Remove unsupported statements.
    2. Correct factual mistakes.
    3. Preserve valid PDF citations.
    4. Do not add outside knowledge.
    5. If the answer is not supported, return:
       No relevant answer found in the PDF.

    PDF Context:
    {context}

    User Question:
    {question}

    Generated Answer:
    {draft}

    Return only the verified final answer.
    """

    response = llm.invoke(prompt)

    return extract_text(response)


# MAIN MULTI-AGENT RAG PIPELINE

def multi_agent_rag(question):

    print("\n========== AGENT 1: QUERY ANALYZER ==========")

    queries = query_analyzer(question)

    print("Generated Queries:")
    for query in queries:
        print("-", query)


    print("\n========== AGENT 2: DOCUMENT RETRIEVER ==========")

    documents = document_retriever(queries)

    print("Retrieved Chunks:", len(documents))

    if not documents:
        return "No relevant answer found in the PDF."

    context = prepare_context(documents)


    print("\n========== AGENT 3: ANSWER GENERATOR ==========")

    draft = answer_generator(question, context)

    print("Generated Answer:")
    print(draft)


    print("\n========== AGENT 4: ANSWER REVIEWER ==========")

    final_answer = answer_reviewer(
        question,
        context,
        draft
    )

    print("\nVerified Answer:")
    print(final_answer)

    return final_answer


print("Multi-Agent RAG System Created Successfully!")


Multi-Agent RAG System Created Successfully!


In [7]:
print("Multi-Agent RAG Chatbot Started!")
print("Ask questions about your uploaded PDF.")
print("Type 'exit' to stop.\n")

while True:

    question = input("You: ").strip()

    if question.lower() == "exit":
        print("Chatbot Stopped. Goodbye!")
        break

    if not question:
        print("Please enter a question.")
        continue

    try:
        answer = multi_agent_rag(question)

        print("\nFINAL RESPONSE:")
        print(answer)
        print("\n" + "=" * 60)

    except Exception as e:
        print("Error:", str(e))
        print("Please check your API key, quota and connection.")

Multi-Agent RAG Chatbot Started!
Ask questions about your uploaded PDF.
Type 'exit' to stop.

You: What is inheritance in Python?

========== AGENT 1: QUERY ANALYZER ==========
Generated Queries:
- Python inheritance explained pdf
- Python OOP class inheritance filetype pdf
- Object oriented programming Python inheritance tutorial pdf

========== AGENT 2: DOCUMENT RETRIEVER ==========
Retrieved Chunks: 4

========== AGENT 3: ANSWER GENERATOR ==========
Generated Answer:
Inheritance allows a new class, called a child class, to reuse or extend features of an existing parent class [python_programming_rag_sample.pdf p.2]. It helps reduce duplicated code by allowing the child class to add new methods or override methods inherited from its parent [python_programming_rag_sample.pdf p.2].

========== AGENT 4: ANSWER REVIEWER ==========

Verified Answer:
Inheritance allows a new class, called a child class, to reuse or extend features of an existing parent class [python_programming_rag_sample.p